In [ ]:
!pip install -q langchain langchain-openai langchain-community langgraph openai langchain-together

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 8.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import os
from google.colab import userdata
os.environ["TOGETHER_API_KEY"] = userdata.get('TOGETHER_API_KEY')

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"]  = userdata.get("OPENROUTER_API_KEY")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

# Free models on OpenRouter — both support tool calling
#MODEL = "openai/gpt-oss-120b:free"
MODEL = "meta-llama/llama-3.3-70b-instruct:free"   # alternative

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model=MODEL, temperature=0)
print("Model :", MODEL)
print("Status: Ready")

Model : meta-llama/llama-3.3-70b-instruct:free
Status: Ready


In [ ]:
#llm
from langchain_together import ChatTogether
llm = ChatTogether(
    model="meta-llama/Llama-3.3-70B-Instruct-Turbo",
    max_tokens=256,
    temperature=0.1
)

In [ ]:
#H.W: what if the city does not exist in the data
from langchain_core.tools import tool #: wrapper

@tool
def get_weather(city:  str) -> str:
  """
  use this for getting the weather for a given city
  """

  data = {
        "Bangalore": "28°C, partly cloudy",
        "Mumbai":    "32°C, humid and hot",
        "Delhi":     "35°C, sunny and hazy",
        "Chennai":   "30°C, warm and breezy",
    }
  return data[city]


@tool
def calculator(expression: str) -> str:
  """Evaluate mathematical expressions and python code. Example '12 * 6 + 5'"""
  result = eval(expression)
  return f"{expression} = {result}"

@tool
def unit_converter(value: float, from_unit: str, to_unit: str) -> str:
    """Convert between units: km<->miles, kg<->lbs, celsius<->fahrenheit"""
    conversions = {
        ("km","miles"):            lambda x: x * 0.621371,
        ("miles","km"):            lambda x: x * 1.60934,
        ("kg","lbs"):              lambda x: x * 2.20462,
        ("celsius","fahrenheit"):  lambda x: x * 9/5 + 32,
        ("fahrenheit","celsius"):  lambda x: (x - 32) * 5/9,
    }
    key = (from_unit.lower(), to_unit.lower())
    result = conversions[key](value) if key in conversions else None
    return f"{value} {from_unit} = {result:.2f} {to_unit}" if result else "Unsupported conversion"

tools = [get_weather, calculator, unit_converter]

In [ ]:
x = "12 + 5"
eval(x) # python code evaluate

17

In [ ]:
x = "sum([1,2,3])" # python code evaluate
eval(x)

6

In [ ]:
llm_with_tools = llm.bind_tools(tools)
type(llm_with_tools)

langchain_core.language_models.chat_models._ChatModelBinding

In [ ]:
response = llm_with_tools.invoke("What's the temperate in mumbai ?")

In [ ]:
print(response.tool_calls)

[{'name': 'get_weather', 'args': {'city': 'Mumbai'}, 'id': 'call_rghekqie68776umvi6epar0u', 'type': 'tool_call'}]


In [ ]:
response = llm_with_tools.invoke(" convert 50 degree celcius to farenhiet")
print(response.tool_calls)

[{'name': 'unit_converter', 'args': {'from_unit': 'celsius', 'to_unit': 'fahrenheit', 'value': 50}, 'id': 'chatcmpl-tool-a24e1275984b7535', 'type': 'tool_call'}]


In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

In [ ]:
agent = create_react_agent(llm, tools)

query = "What is the weather in Delhi? Also convert 100 km to miles."

result = agent.invoke({'messages': query})

/tmp/ipykernel_1823/4173307272.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


In [ ]:
result["messages"][-1].content

'The current weather in Delhi is **35\u202f°C**, sunny and hazy.\n\nAnd **100\u202fkm** is equal to **approximately 62.14\u202fmiles**.'

In [ ]:
for msg in result["messages"]:
  print(msg.content)
  if hasattr(msg, 'tool_calls'):
    print(msg.tool_calls)

What is the weather in Delhi? Also convert 100 km to miles.

[{'name': 'get_weather', 'args': {'city': 'Delhi'}, 'id': 'chatcmpl-tool-b7303d013ece31f0', 'type': 'tool_call'}]
35°C, sunny and hazy

[{'name': 'unit_converter', 'args': {'from_unit': 'km', 'to_unit': 'miles', 'value': 100}, 'id': 'chatcmpl-tool-a4224305a14be992', 'type': 'tool_call'}]
100.0 km = 62.14 miles
The current weather in Delhi is **35 °C**, sunny and hazy.

And **100 km** is equal to **approximately 62.14 miles**.
[]


In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage

In [ ]:
checkpointer = MemorySaver()
agent_mem = create_react_agent(llm, tools, checkpointer = checkpointer, prompt="You are a helpful assistant. Use emojis in output where possible")

NameError: name 'MemorySaver' is not defined

In [ ]:
config = {'configurable': {'thread_id': 'session_1'}}

turns = [
    "Hi! My name is Karan and I work as a AI Engineer.",
    "What is my name and role?",
    "What's the weather in Bangalore?",
]

for query in turns:
  result = agent_mem.invoke({'messages': query}, config = config)
  answer = result['messages'][-1].content
  print(f"User  : {query}")
  print(f"Agent : {answer}")
  print("-" * 50)

User  : Hi! My name is Karan and I work as a AI Engineer.
Agent : Hey Karan! 👋 Great to meet an AI Engineer. How can I help you today? 🚀
--------------------------------------------------
User  : What is my name and role?
Agent : Your name is **Karan**, and you work as an **AI Engineer**. 😊
--------------------------------------------------
User  : What's the weather in Bangalore?
Agent : In Bangalore it’s currently about **28 °C** with partly cloudy skies. 🌤️ Let me know if you need a forecast or anything else!
--------------------------------------------------


In [ ]:
# using non-memory agent
for query in turns:
  result = agent.invoke({'messages': query})
  answer = result['messages'][-1].content
  print(f"User  : {query}")
  print(f"Agent : {answer}")
  print("-" * 50)

User  : Hi! My name is Karan and I work as a AI Engineer.
Agent : Hi Karan! 👋 Great to meet you. How can I help you today?
--------------------------------------------------
User  : What is my name and role?
Agent : I don’t have any information about your name or role. If you’d like me to address you a certain way or tailor my responses to a particular role (e.g., student, developer, manager, etc.), just let me know!
--------------------------------------------------
User  : What's the weather in Bangalore?
Agent : The current weather in Bangalore is about **28 °C** with partly cloudy skies.
--------------------------------------------------


# H.W: run and modify if necessary

In [ ]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage

# Agent with all three tools + memory
full_agent = create_react_agent(
    llm,
    tools,                            # get_weather, calculator, unit_converter
    checkpointer=MemorySaver(),       # short-term memory
    prompt="You are a smart assistant. Use tools when needed."
)

config = {"configurable": {"thread_id": "final_demo"}}

queries = [
    "My name is Karan and I live in Bangalore.",
    "What's the weather where I live? Also, what is 256 * 16?",
    "Convert 42 celsius to fahrenheit.",
    "What's my name and what have we discussed so far?",
]

for q in queries:
    result = full_agent.invoke({"messages": [HumanMessage(q)]}, config=config)
    print(f"User  : {q}")
    print(f"Agent : {result['messages'][-1].content}")
    print("-" * 55)